# Site Filtering: High Flood Risk & Data Quality

Goal: Filter Missouri Basin sites to keep only those with:
1. High streamflow variation (flood risk)
2. Good data quality (low null rates)

This analysis determines which sites to keep in the main `flood_model` table.

In [1]:
import polars as pl
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import wandb
import numpy as np

In [2]:
# Load the flood model dataset from wandb
api = wandb.Api()
artifact = api.artifact("flood-forecasting/flood-dataset:latest")
artifact_dir = artifact.download()

df = pl.read_parquet(f"{artifact_dir}/flood_model.parquet")
print(f"Full dataset: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Total sites: {df['site_id'].n_unique()}")
print(f"Date range: {df['observation_hour'].min()} to {df['observation_hour'].max()}")

wandb: [wandb.Api()] Loaded credentials for https://api.wandb.ai from C:\Users\sacha\_netrc.
wandb: Downloading large artifact 'flood-dataset:latest', 4522.32MB. 1 files...
wandb:   1 of 1 files downloaded.  
Done. 00:00:00.4 (12597.0MB/s)


Full dataset: 101,651,130 rows x 51 columns
Total sites: 1029
Date range: 2007-10-28 05:00:00+00:00 to 2026-02-03 18:00:00+00:00


## 1. Analyze Streamflow Variation Per Site

We compute the coefficient of variation (CV) of streamflow for each site.
CV = std / mean. Higher CV means more variable flow, i.e., higher flood risk.

In [3]:
# Clean bad values: sentinels (-999999), polluted averages, and impossible negatives
df = df.with_columns(
    pl.when(pl.col("streamflow_cfs_mean") < 0).then(None).otherwise(pl.col("streamflow_cfs_mean")).alias("streamflow_cfs_mean"),
    pl.when(pl.col("gage_height_ft_mean") < -100).then(None).otherwise(pl.col("gage_height_ft_mean")).alias("gage_height_ft_mean"),
)
print(f"Cleaned bad values: streamflow < 0 → null, gage height < -100 → null")

# Compute streamflow statistics per site
site_stats = df.group_by("site_id").agg(
    pl.col("streamflow_cfs_mean").mean().alias("streamflow_mean"),
    pl.col("streamflow_cfs_mean").std().alias("streamflow_std"),
    pl.col("streamflow_cfs_mean").max().alias("streamflow_max"),
    pl.col("streamflow_cfs_mean").min().alias("streamflow_min"),
    pl.col("gage_height_ft_mean").mean().alias("gage_height_mean"),
    pl.col("gage_height_ft_mean").std().alias("gage_height_std"),
    pl.col("latitude").first(),
    pl.col("longitude").first(),
    pl.col("station_name").first(),
    pl.len().alias("total_rows"),
    pl.col("streamflow_cfs_mean").null_count().alias("streamflow_nulls"),
    pl.col("gage_height_ft_mean").null_count().alias("gage_height_nulls"),
    pl.col("precipitation_mm").null_count().alias("precip_nulls"),
    pl.col("temperature_c").null_count().alias("temp_nulls"),
)

# Compute coefficient of variation and null rates
site_stats = site_stats.with_columns(
    (pl.col("streamflow_std") / pl.col("streamflow_mean")).alias("streamflow_cv"),
    (pl.col("streamflow_max") - pl.col("streamflow_min")).alias("streamflow_range"),
    (pl.col("streamflow_nulls") / pl.col("total_rows") * 100).alias("streamflow_null_pct"),
    (pl.col("gage_height_nulls") / pl.col("total_rows") * 100).alias("gage_height_null_pct"),
    (pl.col("precip_nulls") / pl.col("total_rows") * 100).alias("precip_null_pct"),
    (pl.col("temp_nulls") / pl.col("total_rows") * 100).alias("temp_null_pct"),
)

print(f"Total sites: {len(site_stats)}")
print(f"Sites with streamflow data (non-null CV): {site_stats.filter(pl.col('streamflow_cv').is_not_null()).shape[0]}")
print(f"Sites with 100% null streamflow (no data): {site_stats.filter(pl.col('streamflow_null_pct') == 100).shape[0]}")
print(f"Sites with zero mean streamflow: {site_stats.filter(pl.col('streamflow_mean') == 0).shape[0]}")

# Remove sites with no data or zero mean (zero mean causes inf CV)
site_stats = site_stats.filter(
    pl.col("streamflow_cv").is_not_null()
    & pl.col("streamflow_cv").is_finite()
    & (pl.col("streamflow_mean") > 0)
)
print(f"Sites after removing no-data and zero-mean: {len(site_stats)}")

# Show top 20 sites by CV
print("\nTop 20 sites by streamflow variation (highest flood risk):")
site_stats.select(
    "site_id", "station_name", "streamflow_cv", "streamflow_mean",
    "streamflow_range", "total_rows", "streamflow_null_pct", "gage_height_null_pct"
).sort("streamflow_cv", descending=True).head(20)

Cleaned bad values: streamflow < 0 → null, gage height < -100 → null
Total sites: 1029
Sites with streamflow data (non-null CV): 939
Sites with 100% null streamflow (no data): 90
Sites with zero mean streamflow: 2
Sites after removing no-data and zero-mean: 937

Top 20 sites by streamflow variation (highest flood risk):


site_id,station_name,streamflow_cv,streamflow_mean,streamflow_range,total_rows,streamflow_null_pct,gage_height_null_pct
str,str,f64,f64,f64,u32,f64,f64
"""06930015""","""McCourtney Hollow Trib at FLW(…",41.26826,0.04586,142.125,15428,0.0,0.155561
"""06846500""","""BEAVER C AT CEDAR BLUFFS, KS""",29.155321,0.280956,817.0,144971,1.360962,0.19935
"""06860000""","""SMOKY HILL R AT ELKADER, KS""",22.586861,1.944389,7985.0,145556,2.46984,0.225343
"""06928380""","""Upper Smith Branch bl Eng. Pon…",21.956005,5.152149,8088.181818,14613,0.0,0.328475
"""06846000""","""BEAVER C AT LUDELL, KS""",19.624369,0.196528,123.5,25531,0.0,100.0
…,…,…,…,…,…,…,…
"""06400875""","""HORSEHEAD CREEK AT OELRICHS,SD""",9.80207,10.247314,6955.0,130774,5.026993,67.729824
"""06860900""","""HACKBERRY C NR TREGO CENTER, K…",9.794033,2.403507,1972.5,83218,1.101925,0.133385
"""06711570""","""HARVARD GULCH AT COLORADO BLVD…",9.588001,0.637842,542.688333,58331,0.0,73.775522


In [ ]:
# Distribution of streamflow CV
cv_data = site_stats.to_pandas()
median_cv = cv_data["streamflow_cv"].median()

# Remove extreme outliers from plot data so bins spread evenly
p95 = cv_data["streamflow_cv"].quantile(0.95)
cv_clipped = cv_data[cv_data["streamflow_cv"] <= p95].copy()
n_outliers = len(cv_data) - len(cv_clipped)

fig = make_subplots(rows=2, cols=1, row_heights=[0.7, 0.3],
    subplot_titles=(
        f"Histogram of Streamflow CV ({len(cv_data)} sites, {n_outliers} outliers clipped)",
        "Box Plot (full range, shows outliers)"
    ))

fig.add_trace(
    go.Histogram(x=cv_clipped["streamflow_cv"], nbinsx=30,
                 name="Sites", marker_color="steelblue"),
    row=1, col=1
)
fig.add_vline(x=median_cv, line_dash="dash", line_color="red",
              annotation_text=f"Median: {median_cv:.2f}", row=1, col=1)

fig.add_trace(
    go.Box(x=cv_data["streamflow_cv"], name="CV", marker_color="steelblue"),
    row=2, col=1
)

fig.update_layout(height=500, showlegend=False, bargap=0.05)
fig.update_xaxes(title_text="CV (std/mean)", row=1, col=1)
fig.update_yaxes(title_text="Number of Sites", row=1, col=1)
fig.show()

print(f"CV range: {cv_data['streamflow_cv'].min():.2f} to {cv_data['streamflow_cv'].max():.2f}")
print(f"Median CV: {median_cv:.2f}")
print(f"Sites above median: {len(cv_data[cv_data['streamflow_cv'] > median_cv])}")
print(f"Sites below median: {len(cv_data[cv_data['streamflow_cv'] <= median_cv])}")
print(f"Outliers clipped from histogram (CV > {p95:.2f}): {n_outliers}")

## 2. Analyze Null Rates Per Site

In [ ]:
# Distribution of null rates
null_data = site_stats.to_pandas()

fig = make_subplots(rows=1, cols=2,
    subplot_titles=("Streamflow Null %", "Gage Height Null %"))

fig.add_trace(
    go.Histogram(x=null_data["streamflow_null_pct"], nbinsx=50, name="Streamflow"),
    row=1, col=1
)
fig.add_trace(
    go.Histogram(x=null_data["gage_height_null_pct"], nbinsx=50, name="Gage Height"),
    row=1, col=2
)
fig.update_layout(title="Distribution of Null Rates Across Sites", showlegend=False)
fig.show()

# Summary
print(f"Sites with 0% streamflow nulls:    {len(null_data[null_data['streamflow_null_pct'] == 0])}")
print(f"Sites with <20% streamflow nulls:  {len(null_data[null_data['streamflow_null_pct'] < 20])}")
print(f"Sites with >50% streamflow nulls:  {len(null_data[null_data['streamflow_null_pct'] > 50])}")
print(f"Sites with 100% streamflow nulls:  {len(null_data[null_data['streamflow_null_pct'] == 100])}")
print()
print(f"Sites with 0% gage height nulls:   {len(null_data[null_data['gage_height_null_pct'] == 0])}")
print(f"Sites with <20% gage height nulls: {len(null_data[null_data['gage_height_null_pct'] < 20])}")
print(f"Sites with >50% gage height nulls: {len(null_data[null_data['gage_height_null_pct'] > 50])}")
print(f"Sites with 100% gage height nulls: {len(null_data[null_data['gage_height_null_pct'] == 100])}")

Sites with 0% streamflow nulls:    149
Sites with <20% streamflow nulls:  810
Sites with >50% streamflow nulls:  7
Sites with 100% streamflow nulls:  0

Sites with 0% gage height nulls:   30
Sites with <20% gage height nulls: 616
Sites with >50% gage height nulls: 289
Sites with 100% gage height nulls: 139


## 3. Apply Filters

Criteria:
- Streamflow CV > median (above-average variation = higher flood risk)
- Gage height null rate < 20% (good data quality)
- Streamflow null rate < 20% (good data quality)

Adjust thresholds based on the distributions above.

In [ ]:
# Set thresholds (adjust after reviewing distributions above)
CV_THRESHOLD = site_stats.filter(
    pl.col("streamflow_cv").is_not_null()
)["streamflow_cv"].median()
NULL_THRESHOLD = 20  # max % nulls allowed

print(f"CV threshold (median): {CV_THRESHOLD:.2f}")
print(f"Null threshold: {NULL_THRESHOLD}%")

# Apply filters
filtered_sites = site_stats.filter(
    (pl.col("streamflow_cv") > CV_THRESHOLD)
    & (pl.col("streamflow_null_pct") < NULL_THRESHOLD)
    & (pl.col("gage_height_null_pct") < NULL_THRESHOLD)
)

print(f"\nBefore filtering: {len(site_stats)} sites")
print(f"After filtering:  {len(filtered_sites)} sites")
print(f"Removed:          {len(site_stats) - len(filtered_sites)} sites")

# Breakdown of why sites were removed
no_data = site_stats.filter(pl.col("streamflow_cv").is_null())
low_cv = site_stats.filter(
    (pl.col("streamflow_cv").is_not_null()) & (pl.col("streamflow_cv") <= CV_THRESHOLD)
)
high_nulls = site_stats.filter(
    (pl.col("streamflow_cv") > CV_THRESHOLD)
    & ((pl.col("streamflow_null_pct") >= NULL_THRESHOLD) | (pl.col("gage_height_null_pct") >= NULL_THRESHOLD))
)
print(f"\nRemoval breakdown:")
print(f"  No streamflow data at all: {len(no_data)}")
print(f"  Low variation (CV <= {CV_THRESHOLD:.2f}): {len(low_cv)}")
print(f"  High nulls (>= {NULL_THRESHOLD}%): {len(high_nulls)}")

CV threshold (median): 1.89
Null threshold: 20%

Before filtering: 937 sites
After filtering:  266 sites
Removed:          671 sites

Removal breakdown:
  No streamflow data at all: 0
  Low variation (CV <= 1.89): 469
  High nulls (>= 20%): 202


## 4. Visualizations: Before vs After

In [ ]:
# Map: All sites vs Filtered sites
all_sites_pd = site_stats.to_pandas()
all_sites_pd["status"] = "Removed"
kept_ids = set(filtered_sites["site_id"].to_list())
all_sites_pd.loc[all_sites_pd["site_id"].isin(kept_ids), "status"] = "Kept"

fig = px.scatter_geo(
    all_sites_pd,
    lat="latitude", lon="longitude",
    color="status",
    color_discrete_map={"Kept": "blue", "Removed": "red"},
    hover_name="station_name",
    hover_data=["site_id", "streamflow_cv", "streamflow_null_pct", "gage_height_null_pct"],
    title=f"Site Filtering: {len(filtered_sites)} Kept (blue) vs {len(site_stats) - len(filtered_sites)} Removed (red)",
    scope="usa",
)
fig.update_layout(geo=dict(center=dict(lat=45, lon=-105), projection_scale=3))
fig.show()

In [ ]:
# Scatter: CV vs Null Rate (shows filtering logic)
# Only show sites that have streamflow data (non-null CV)
scatter_data = all_sites_pd.copy()

fig = px.scatter(
    scatter_data,
    x="streamflow_cv", y="gage_height_null_pct",
    color="status",
    color_discrete_map={"Kept": "blue", "Removed": "red"},
    hover_name="station_name",
    hover_data=["site_id"],
    title="Streamflow Variation vs Data Quality (sites with data only)",
    labels={"streamflow_cv": "Streamflow CV (higher = more variable)",
            "gage_height_null_pct": "Gage Height Null %"},
)
fig.add_vline(x=CV_THRESHOLD, line_dash="dash", line_color="gray",
              annotation_text=f"CV threshold: {CV_THRESHOLD:.2f}")
fig.add_hline(y=NULL_THRESHOLD, line_dash="dash", line_color="gray",
              annotation_text=f"Null threshold: {NULL_THRESHOLD}%")
fig.show()

In [ ]:
# Comparison: dataset size before vs after (CV/null filter only; next cell restricts to Missouri)
kept_df = df.filter(pl.col("site_id").is_in(filtered_sites["site_id"].implode().first()))

print(f"Rows before: {len(df):,}")
print(f"Rows after:  {len(kept_df):,}")
print(f"Reduction:   {(1 - len(kept_df)/len(df))*100:.1f}%")
print(f"\nSites before: {df['site_id'].n_unique()}")
print(f"Sites after:  {kept_df['site_id'].n_unique()}")

Rows before: 101,651,130
Rows after:  30,211,684
Reduction:   70.3%

Sites before: 1029
Sites after:  266


In [ ]:
# Restrict kept_df to sites in Missouri (state) using state boundary polygon
# Census TIGER state boundaries (STATEFP 29 = Missouri) — excludes Kansas, etc.

import geopandas as gpd
from shapely.geometry import Point

# Load US state boundaries (zip URL); STATEFP 29 = Missouri; to_crs(4326) = WGS84 for lat/lon
states_url = "https://www2.census.gov/geo/tiger/GENZ2018/shp/cb_2018_us_state_20m.zip"
states = gpd.read_file(states_url)
mo = states[states.STATEFP == "29"].to_crs(4326).geometry.iloc[0]

# Restrict to sites that passed the CV/null filter; use implode().first() to avoid is_in deprecation
kept_sites = site_stats.filter(
    pl.col("site_id").is_in(filtered_sites["site_id"].implode().first())
)
# For each kept site, build (lon, lat) and test if point is inside Missouri polygon
in_mo = kept_sites.select("site_id", "latitude", "longitude").with_columns(
    pl.struct("longitude", "latitude")
    .map_elements(
        lambda r: Point(r["longitude"], r["latitude"]).within(mo),
        return_dtype=pl.Boolean,
    )
    .alias("in_missouri")
)
# List of site_ids that lie inside the Missouri state boundary
missouri_site_ids = in_mo.filter(pl.col("in_missouri"))["site_id"].to_list()
# Restrict kept_df to only those sites (one row per site-hour, so filter by site_id)
kept_df = kept_df.filter(pl.col("site_id").is_in(missouri_site_ids))
print(f"After keeping only Missouri sites (state polygon): {len(kept_df):,} rows, {kept_df['site_id'].n_unique()} sites")

After keeping only Missouri sites (state polygon): 10,545,056 rows, 82 sites


In [ ]:
# Map: Kept sites only — Missouri (blue) vs not in Missouri (red), by state boundary polygon
# Reuse Missouri polygon from previous cell if available, else load Census TIGER boundary
try:
    _ = mo
except NameError:
    import geopandas as gpd
    states_url = "https://www2.census.gov/geo/tiger/GENZ2018/shp/cb_2018_us_state_20m.zip"
    states = gpd.read_file(states_url)
    mo = states[states.STATEFP == "29"].to_crs(4326).geometry.iloc[0]
from shapely.geometry import Point

# Same set of kept sites as above; use implode().first() to avoid is_in deprecation
kept_sites_for_map = site_stats.filter(
    pl.col("site_id").is_in(filtered_sites["site_id"].implode().first())
).with_columns(
    pl.struct("longitude", "latitude")
    .map_elements(
        lambda r: Point(r["longitude"], r["latitude"]).within(mo),
        return_dtype=pl.Boolean,
    )
    .alias("in_missouri")
).with_columns(
    pl.when(pl.col("in_missouri")).then(pl.lit("Missouri")).otherwise(pl.lit("Not Missouri")).alias("status")
)

kept_sites_pd = kept_sites_for_map.to_pandas()
n_mo = (kept_sites_for_map["status"] == "Missouri").sum()
n_not_mo = (kept_sites_for_map["status"] == "Not Missouri").sum()

fig = px.scatter_geo(
    kept_sites_pd,
    lat="latitude",
    lon="longitude",
    color="status",
    color_discrete_map={"Missouri": "blue", "Not Missouri": "red"},
    hover_name="station_name",
    hover_data=["site_id", "streamflow_cv", "streamflow_null_pct", "gage_height_null_pct"],
    title=f"Kept sites: {n_mo} in Missouri (blue) vs {n_not_mo} outside Missouri (red)",
    scope="usa",
)
fig.update_layout(geo=dict(center=dict(lat=45, lon=-105), projection_scale=3))
fig.show()

In [ ]:
# Export the final list of kept site IDs (Missouri only) as a dbt seed CSV
# kept_df is already Missouri-only; this seed feeds flood_model_filtered
from pathlib import Path

kept_site_ids = kept_df.select("site_id").unique().sort("site_id")
print(f"Kept {len(kept_site_ids)} site IDs (Missouri only)")

seed_path = Path("../elt/transformation/seeds/filtered_site_ids.csv")
kept_site_ids.write_csv(seed_path)
print(f"Saved to {seed_path.resolve()}")
print(kept_site_ids)

Kept 266 site IDs
Saved to C:\Users\sacha\RiceLocal\Capstone\Coding\Flood-Forecasting\elt\transformation\seeds\filtered_site_ids.csv
shape: (266, 1)
┌─────────────────┐
│ site_id         │
│ ---             │
│ str             │
╞═════════════════╡
│ 06062500        │
│ 06101200        │
│ 06119600        │
│ 06123030        │
│ 06127500        │
│ …               │
│ 06935980        │
│ 06935997        │
│ 06936475        │
│ 06936530        │
│ 411450095582201 │
└─────────────────┘


In [ ]:
# Check for suspicious sentinel values in streamflow and gage height
print("Streamflow stats:")
print(f"  Min: {kept_df['streamflow_cfs_mean'].min()}")
print(f"  Max: {kept_df['streamflow_cfs_mean'].max()}")
print(f"  Rows >= 99999: {kept_df.filter(pl.col('streamflow_cfs_mean') >= 999999).shape[0]}")
print(f"  Rows <= -99999: {kept_df.filter(pl.col('streamflow_cfs_mean') <= -999999).shape[0]}")

print("\nGage height stats:")
print(f"  Min: {kept_df['gage_height_ft_mean'].min()}")
print(f"  Max: {kept_df['gage_height_ft_mean'].max()}")
print(f"  Rows >= 99999: {kept_df.filter(pl.col('gage_height_ft_mean') >= 999999).shape[0]}")
print(f"  Rows <= -99999: {kept_df.filter(pl.col('gage_height_ft_mean') <= -999999).shape[0]}")

print("\nNegative streamflow rows:", kept_df.filter(pl.col("streamflow_cfs_mean") < 0).shape[0])
print("Negative gage height rows:", kept_df.filter(pl.col("gage_height_ft_mean") < 0).shape[0])

Streamflow stats:
  Min: 0.0
  Max: 197000.0
  Rows >= 99999: 0
  Rows <= -99999: 0

Gage height stats:
  Min: -1.38
  Max: 73.2825
  Rows >= 99999: 0
  Rows <= -99999: 0

Negative streamflow rows: 0
Negative gage height rows: 155416
